# CSCI 347 Project 2
## Problem 1 - Dataset Familiarization

Dataset chosen: **Twitch Social Networks (ENGB split)**

Dataset link: https://snap.stanford.edu/data/twitch-social-networks.html

### 1) Why did you choose this dataset, and why is it interesting to you? (Make sure you include a link to the dataset that you chose).

### 2) Did you preprocess the data? If so, how did you preprocess it? Clearly explain the process. e.g., Did you take the largest connected component? Was it manageable, if not, did you sample it? How did you take the sample of the graph? What did you do after sampling? Did you do any other preprocessing? etc. 

### 3) Explain what type of data is represented in this dataset.

### 4a)  What characteristic do you expect the nodes with high centrality to possess? E.g., what type of non-graph characteristic would you expect the nodes with high centrality have? Note that this answer depends on the dataset that you choose.

### 4b) Do you think this graph dataset would exhibit power law? State your reasons.

### 4c) Do you expect this graph to have small-world property? State your reasons.

## Problem 2 - Function Implementations

The following cells implement all 12 required functions for a simple, undirected, unweighted graph using an edge-list input.


In [ ]:
from collections import defaultdict, deque
import numpy as np

def load_edge_list_txt(path):
    """Load whitespace-separated edge list file into list[tuple[int,int]]."""
    edges = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            u, v = line.split()[:2]
            edges.append((int(u), int(v)))
    return edges

def _build_adj(edges):
    adj = defaultdict(set)
    for u, v in edges:
        if u == v:
            continue
        adj[u].add(v)
        adj[v].add(u)
    return adj

def _bfs_distances(adj, source):
    if source not in adj:
        return {source: 0}
    dist = {source: 0}
    q = deque([source])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

# 1) Number of nodes
def number_of_nodes(edges):
    nodes = set()
    for u, v in edges:
        nodes.add(u)
        nodes.add(v)
    return len(nodes)

# 2) Degree of a vertex
def degree_of_vertex(edges, vertex):
    adj = _build_adj(edges)
    return len(adj.get(vertex, set()))

# 3) Adjacency matrix
def adjacency_matrix(edges, return_nodes=False):
    nodes = sorted({x for e in edges for x in e})
    idx = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    A = np.zeros((n, n), dtype=int)
    for u, v in edges:
        i, j = idx[u], idx[v]
        A[i, j] = 1
        A[j, i] = 1
    if return_nodes:
        return A, nodes
    return A

# 4) Degree distribution as list of counts
def degree_distribution(edges):
    adj = _build_adj(edges)
    if not adj:
        return []
    degs = [len(adj[u]) for u in adj]
    max_deg = max(degs)
    dist = [0] * (max_deg + 1)
    for d in degs:
        dist[d] += 1
    return dist

# 5) P( degree >= k )
def prob_degree_geq(deg_dist, k):
    total = sum(deg_dist)
    if total == 0:
        return 0.0
    if k <= 0:
        return 1.0
    if k >= len(deg_dist):
        return 0.0
    return sum(deg_dist[k:]) / total

# 6) Eccentricity of a vertex
def eccentricity_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nodes = set(adj.keys())
    if vertex not in nodes:
        return float("inf")
    dist = _bfs_distances(adj, vertex)
    if len(dist) != len(nodes):
        return float("inf")
    return max(dist.values())

# 7) Diameter of graph
def diameter_of_graph(edges):
    adj = _build_adj(edges)
    nodes = list(adj.keys())
    if not nodes:
        return 0
    diam = 0
    for u in nodes:
        dist = _bfs_distances(adj, u)
        if len(dist) != len(nodes):
            return float("inf")
        ecc = max(dist.values())
        if ecc > diam:
            diam = ecc
    return diam

# 8) Radius of graph
def radius_of_graph(edges):
    adj = _build_adj(edges)
    nodes = list(adj.keys())
    if not nodes:
        return 0
    radius = float("inf")
    for u in nodes:
        dist = _bfs_distances(adj, u)
        if len(dist) != len(nodes):
            return float("inf")
        ecc = max(dist.values())
        if ecc < radius:
            radius = ecc
    return radius

# 9) Clustering coefficient of a vertex
def clustering_coefficient_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nbrs = list(adj.get(vertex, []))
    k = len(nbrs)
    if k < 2:
        return 0.0
    links = 0
    for i in range(k):
        u = nbrs[i]
        for j in range(i + 1, k):
            v = nbrs[j]
            if v in adj[u]:
                links += 1
    return (2 * links) / (k * (k - 1))

# 10) Betweenness centrality of a vertex (Brandes, unweighted graph)
def betweenness_centrality_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nodes = list(adj.keys())
    if vertex not in adj:
        return 0.0

    cb_v = 0.0
    for s in nodes:
        stack = []
        pred = {w: [] for w in nodes}
        sigma = {w: 0 for w in nodes}
        dist = {w: -1 for w in nodes}
        sigma[s] = 1
        dist[s] = 0
        q = deque([s])

        while q:
            v = q.popleft()
            stack.append(v)
            for w in adj[v]:
                if dist[w] < 0:
                    q.append(w)
                    dist[w] = dist[v] + 1
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    pred[w].append(v)

        delta = {w: 0.0 for w in nodes}
        while stack:
            w = stack.pop()
            for v in pred[w]:
                if sigma[w] != 0:
                    delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
            if w != s and w == vertex:
                cb_v += delta[w]

    return cb_v / 2.0

# 11) Closeness centrality of a vertex
def closeness_centrality_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nodes = set(adj.keys())
    if vertex not in nodes:
        return 0.0
    dist = _bfs_distances(adj, vertex)
    if len(dist) != len(nodes):
        return 0.0
    total = sum(dist.values())
    if total == 0:
        return 0.0
    return (len(nodes) - 1) / total

# 12) Eigenvector centrality with power iteration and L2 normalization
def eigenvector_centrality_power_iteration(A, max_iter=200, tol=1e-9):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    x = np.ones(n, dtype=float)
    x /= np.linalg.norm(x, ord=2)

    for _ in range(max_iter):
        x_new = A @ x
        norm = np.linalg.norm(x_new, ord=2)
        if norm == 0:
            return x_new
        x_new /= norm
        if np.linalg.norm(x_new - x, ord=2) < tol:
            x = x_new
            break
        x = x_new
    return x


### Problem 2 Tests on `facebook_combined`

This cell runs each function at least once on SNAP `facebook_combined.txt`.


In [ ]:
import os

candidate_paths = [
    "data/facebook_combined.txt",
    "Data/facebook_combined.txt"
]
facebook_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if facebook_path is None:
    raise FileNotFoundError("Could not find facebook_combined.txt in expected data folders.")

print("Using dataset:", facebook_path)
fb_edges = load_edge_list_txt(facebook_path)
test_vertex = 0

print("#1 Number of nodes:", number_of_nodes(fb_edges))
print("#2 Degree of vertex", test_vertex, ":", degree_of_vertex(fb_edges, test_vertex))

A_fb, fb_nodes = adjacency_matrix(fb_edges, return_nodes=True)
print("#3 Adjacency matrix shape:", A_fb.shape)

deg_dist = degree_distribution(fb_edges)
print("#4 Degree distribution length:", len(deg_dist), "(max degree + 1)")

k = 10
print(f"#5 P(degree >= {k}):", prob_degree_geq(deg_dist, k))

print("#6 Eccentricity of vertex", test_vertex, ":", eccentricity_of_vertex(test_vertex, fb_edges))
print("#7 Graph diameter:", diameter_of_graph(fb_edges))
print("#8 Graph radius:", radius_of_graph(fb_edges))

print("#9 Clustering coefficient of vertex", test_vertex, ":", clustering_coefficient_of_vertex(test_vertex, fb_edges))
print("#10 Betweenness centrality of vertex", test_vertex, ":", betweenness_centrality_of_vertex(test_vertex, fb_edges))
print("#11 Closeness centrality of vertex", test_vertex, ":", closeness_centrality_of_vertex(test_vertex, fb_edges))

eig = eigenvector_centrality_power_iteration(A_fb, max_iter=200)
top_idx = np.argsort(eig)[-10:][::-1]
top_nodes = [(fb_nodes[i], float(eig[i])) for i in top_idx]
print("#12 Top 10 nodes by eigenvector centrality (node, score):")
for item in top_nodes:
    print(item)


Using dataset: data/facebook_combined.txt
#1 Number of nodes: 4039
#2 Degree of vertex 0 : 347
#3 Adjacency matrix shape: (4039, 4039)
#4 Degree distribution length: 1046 (max degree + 1)
#5 P(degree >= 10): 0.7858380787323594
#6 Eccentricity of vertex 0 : 6
#7 Graph diameter: 8


## Problem 3 - Analysis Cells

Code-only implementation cells for Problem 3.


In [ ]:
import os
import random
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display

# Resolve dataset paths (data/ first, then Data/ fallback)
edge_candidates = [
    "data/Project_2/twitch/ENGB/musae_ENGB_edges.csv",
    "Data/Project_2/twitch/ENGB/musae_ENGB_edges.csv"
]
target_candidates = [
    "data/Project_2/twitch/ENGB/musae_ENGB_target.csv",
    "Data/Project_2/twitch/ENGB/musae_ENGB_target.csv"
]

edge_path = next((p for p in edge_candidates if os.path.exists(p)), None)
target_path = next((p for p in target_candidates if os.path.exists(p)), None)
if edge_path is None or target_path is None:
    raise FileNotFoundError("Could not find ENGB Twitch files in data/Project_2/twitch/ENGB")

print("Edges file:", edge_path)
print("Target file:", target_path)

edges_df = pd.read_csv(edge_path)
target_df = pd.read_csv(target_path)
u_col, v_col = edges_df.columns[:2]

# Build simple undirected graph
G = nx.Graph()
G.add_edges_from(edges_df[[u_col, v_col]].itertuples(index=False, name=None))
G.remove_edges_from(nx.selfloop_edges(G))

# Largest connected component
largest_cc_nodes = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(largest_cc_nodes).copy()

print("Original graph: nodes=", G.number_of_nodes(), "edges=", G.number_of_edges())
print("Largest CC: nodes=", G_lcc.number_of_nodes(), "edges=", G_lcc.number_of_edges())


### 3.1 Graph Visualization


In [ ]:
# Visualize a sample if graph is large
VIS_SAMPLE_NODES = 400
SEED = 347

if G_lcc.number_of_nodes() > VIS_SAMPLE_NODES:
    random.seed(SEED)
    sampled_nodes = random.sample(list(G_lcc.nodes()), VIS_SAMPLE_NODES)
    G_vis = G_lcc.subgraph(sampled_nodes).copy()
else:
    G_vis = G_lcc

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G_vis, seed=SEED, k=None)
nx.draw_networkx_nodes(G_vis, pos, node_size=18, alpha=0.8)
nx.draw_networkx_edges(G_vis, pos, width=0.3, alpha=0.35)
plt.title(f"Twitch ENGB visualization ({G_vis.number_of_nodes()} sampled nodes)")
plt.axis("off")
plt.show()


### 3.2 to 3.6 Top-10 Rankings


In [ ]:
def top_k_dict(metric_dict, k=10):
    return sorted(metric_dict.items(), key=lambda x: x[1], reverse=True)[:k]

# 3.2 top closeness
closeness_scores = nx.closeness_centrality(G_lcc)
top10_closeness = top_k_dict(closeness_scores, 10)
print("Top 10 by closeness centrality")
display(pd.DataFrame(top10_closeness, columns=["node", "closeness"]))

# 3.3 top betweenness
betweenness_scores = nx.betweenness_centrality(G_lcc, normalized=True)
top10_betweenness = top_k_dict(betweenness_scores, 10)
print("Top 10 by betweenness centrality")
display(pd.DataFrame(top10_betweenness, columns=["node", "betweenness"]))

# 3.4 top clustering coefficient
clustering_scores = nx.clustering(G_lcc)
top10_clustering = top_k_dict(clustering_scores, 10)
print("Top 10 by clustering coefficient")
display(pd.DataFrame(top10_clustering, columns=["node", "clustering_coeff"]))

# 3.5 top eigenvector centrality
eigen_scores = nx.eigenvector_centrality(G_lcc, max_iter=1000, tol=1e-06)
top10_eigen = top_k_dict(eigen_scores, 10)
print("Top 10 by eigenvector centrality")
display(pd.DataFrame(top10_eigen, columns=["node", "eigenvector"]))

# 3.6 top PageRank
pagerank_scores = nx.pagerank(G_lcc, alpha=0.85)
top10_pagerank = top_k_dict(pagerank_scores, 10)
print("Top 10 by PageRank")
display(pd.DataFrame(top10_pagerank, columns=["node", "pagerank"]))


### 3.7 Overlap Comparison (No Written Interpretation)


In [ ]:
rank_sets = {
    "closeness": {n for n, _ in top10_closeness},
    "betweenness": {n for n, _ in top10_betweenness},
    "clustering": {n for n, _ in top10_clustering},
    "eigenvector": {n for n, _ in top10_eigen},
    "pagerank": {n for n, _ in top10_pagerank},
}

names = list(rank_sets.keys())
overlap_counts = pd.DataFrame(index=names, columns=names, dtype=int)
jaccard_scores = pd.DataFrame(index=names, columns=names, dtype=float)

for a in names:
    for b in names:
        inter = rank_sets[a] & rank_sets[b]
        union = rank_sets[a] | rank_sets[b]
        overlap_counts.loc[a, b] = len(inter)
        jaccard_scores.loc[a, b] = len(inter) / len(union) if union else 0.0

print("Top-10 overlap counts")
display(overlap_counts)
print("Top-10 Jaccard similarity")
display(jaccard_scores.round(3))

comparison_rows = []
for a, b in combinations(names, 2):
    inter = sorted(rank_sets[a] & rank_sets[b])
    comparison_rows.append({
        "metric_a": a,
        "metric_b": b,
        "overlap_count": len(inter),
        "overlap_nodes": inter
    })
display(pd.DataFrame(comparison_rows))


### 3.8 Degree Distribution Log-Log Plot + Least-Squares Fit


In [ ]:
degrees = [d for _, d in G_lcc.degree()]
values, counts = np.unique(degrees, return_counts=True)
probs = counts / counts.sum()

# keep positive values for log scale
mask = (values > 0) & (probs > 0)
k = values[mask].astype(float)
f_k = probs[mask].astype(float)

log_k = np.log(k)
log_f = np.log(f_k)

# least-squares line on log-log points
slope, intercept = np.polyfit(log_k, log_f, 1)
fit_line = slope * log_k + intercept

# R^2
ss_res = np.sum((log_f - fit_line) ** 2)
ss_tot = np.sum((log_f - np.mean(log_f)) ** 2)
r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

plt.figure(figsize=(8, 6))
plt.scatter(log_k, log_f, s=24, alpha=0.8, label="data")
plt.plot(log_k, fit_line, color="red", lw=2, label="least-squares fit")
plt.xlabel("log k")
plt.ylabel("log f(k)")
plt.title("Log-Log Degree Distribution (ENGB LCC)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("fit_slope:", float(slope))
print("fit_intercept:", float(intercept))
print("fit_r2:", float(r2))


### Notes Placeholder
- Add your written interpretation for Problem 3.7 and 3.8 here.
